In [3]:

from openai import OpenAI
from map_text import prompt_key_list, resolve, build_search_items
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="mock-local-key"
)

In [4]:
requirement_query = ""

In [4]:
"""
parser_prompt.py

Prompt + parser for turning a house-hunting request into a small JSON spec.
Uses amenity_lookup.py, so the model only picks from keys your code understands.
"""
import json


# Placeholders are swapped with .replace() so the JSON braces never need escaping
# (an f-string would treat every { } in the examples as a variable).
PARSER_PROMPT = """You convert a house-hunting request into JSON.
Reply with one JSON object only. No explanation, no markdown, no code fences.

Fields:
- "area": the place the user wants to live, e.g. "Merton, London". Use null if none is given.
- "amenities": a list of strings, one per thing the user wants nearby or true of the area.
  Each string must be copied exactly from the allowed keys below.
  If nothing fits, use the user's own short word instead.
  Only include what the user asked for.
  if specific brand amenities are requested, use the brand key, not the generic word.
- "max_distance_miles": a number.
  If the user states a distance, use it.
  "walking distance" -> 0.5
  "nearby", "close" or "local" -> 1.0
  Nothing said -> 2.0

Rules:
- Safety wishes ("safe", "low crime", "not dodgy") go in amenities as "safe".
- Good, top or outstanding schools -> "good_school_ofsted". Plain schools -> "school".
- A cuisine replaces the generic word: "vietnamese restaurants" -> "vietnamese", not "restaurant".
- Named chains use their brand key: "Morley's" -> "morleys", "Pepe's Piri Piri" -> "pepes".
- Use lower-case JSON values: true, false, null.

Allowed keys, by group:
<<KEYS>>

Example 1
Request: I want to find a house in Croydon, London with good schools, restaurants and parks nearby.
Answer: {"area": "Croydon, London", "amenities": ["good_school_ofsted", "restaurant", "park"], "max_distance_miles": 1.0}

Example 2
Request: Looking in Merton, London for vietnamese restaurants, Pepe's Piri Piri and Morley's within 0.5 miles, good schools, and most importantly a safe area.
Answer: {"area": "Merton, London", "amenities": ["vietnamese", "pepes", "morleys", "good_school_ofsted", "safe"], "max_distance_miles": 0.5}

Example 3
Request: Somewhere with a tube station and a gym, not too pricey.
Answer: {"area": null, "amenities": ["tube", "gym", "affordable"], "max_distance_miles": 2.0}

Example 4
Request: Zzzz
Answer: {"area": null, "amenities": [], "max_distance_miles": 2.0}

Request: <<QUERY>>
Answer:"""


def build_parser_prompt(query: str) -> str:
    return PARSER_PROMPT.replace("<<KEYS>>", prompt_key_list()).replace("<<QUERY>>", query.strip())


def parse_query(raw: str) -> dict:
    """Turn the model's reply into a clean spec. Never raises; falls back to defaults."""
    default = {"area": None, "amenities": [], "max_distance_miles": 2.0}
    try:
        data = json.loads(raw[raw.index("{"): raw.rindex("}") + 1])
    except ValueError:
        return default
    if not isinstance(data, dict):
        return default

    amenities = []
    for term in data.get("amenities") or []:
        key = resolve(str(term))          # maps synonyms, drops unknown words
        if key and key not in amenities:
            amenities.append(key)

    try:
        dist = float(data.get("max_distance_miles", 2.0))
    except (TypeError, ValueError):
        dist = 2.0

    return {
        "area": data.get("area") or None,
        "amenities": amenities,
        "max_distance_miles": min(max(dist, 0.1), 10.0),
    }

In [5]:
requirement_query = """
I want a house in Lambeth, London with morleys, pepe's piri piri and lidl nearby, good primary schools and within walking distance. It must be a safe area and not too expensive.
"""

response = client.chat.completions.create(
    # model="mlx-community/gemma-4-e4b-it-4bit",

    model = "jedisct1/gemma-4-E2B-it-txt-mlx-4bit",
    # STRICTLY use only 'user' role for Gemma
    messages=[{"role": "user", "content": build_parser_prompt(requirement_query)}],
    max_tokens=2000
)

message = response.choices[0].message

if hasattr(message, 'content') and message.content:
    insights = message.content.strip()
    print(f"content: {insights}")
elif hasattr(message, 'reasoning') and message.reasoning:
    # Local server put the actual text inside the reasoning block
    insights = message.reasoning.strip()
    print(f"reasoning: {insights}")

content: {"area": "Lambeth, London", "amenities": ["morleys", "pepes", "lidl", "school", "safe", "affordable"], "max_distance_miles": 0.5}


In [6]:
insights

'{"area": "Lambeth, London", "amenities": ["morleys", "pepes", "lidl", "school", "safe", "affordable"], "max_distance_miles": 0.5}'

In [ ]:
insights["area"]

In [54]:

from house_functions import load_houses_data, serach, df_prerequists


query = "lambeth, London"
place_query = [x.strip() for x in query.split(",") if x.strip()]
houses_df = load_houses_data("/Users/danishanrahulan/Downloads/house_prices_formated.csv")
house_df_borough_list, house_df_town_list, house_df_county_list, house_df_outward_code_list, house_df_postcode_list = df_prerequists(houses_df)
check = serach(place_query, house_df_borough_list, house_df_town_list, house_df_county_list, house_df_outward_code_list, house_df_postcode_list)

place_query is {'lambeth', 'London'}


In [55]:
check

[{'LAMBETH': [('Borough', 'LAMBETH', 1, None),
   ('Borough', 'LAMBETH', 4, 100.0)]},
 {'LONDON': [('Town/City', 'LONDON', 1, None),
   ('Borough', 'CITY OF LONDON', 2, None),
   ('County', 'GREATER LONDON', 2, None)],
  'total_matches': 1,
  'truncated': True}]

In [45]:
print(longitude_latitude_from_postcode("CR4 3JJ"))

{'postcode': 'CR4 3JJ', 'quality': 1, 'eastings': 527577, 'northings': 169925, 'country': 'England', 'nhs_ha': 'London', 'longitude': -0.166851, 'latitude': 51.413999, 'european_electoral_region': 'London', 'primary_care_trust': 'Sutton and Merton', 'region': 'London', 'lsoa': 'Merton 013E', 'msoa': 'Merton 013', 'incode': '3JJ', 'outcode': 'CR4', 'parliamentary_constituency': 'Mitcham and Morden', 'parliamentary_constituency_2024': 'Mitcham and Morden', 'senedd_constituency': None, 'senedd_constituency_no': None, 'admin_district': 'Merton', 'parish': 'Merton, unparished area', 'admin_county': None, 'date_of_introduction': '198001', 'date_of_termination': None, 'index_of_multiple_deprivation': 11085, 'admin_ward': 'Lavender Fields', 'ced': None, 'ccg': 'NHS South West London', 'nuts': 'Merton', 'pfa': 'Metropolitan Police', 'nhs_region': 'London', 'ttwa': 'London', 'national_park': 'England (non-National Park)', 'bua': 'Merton', 'icb': 'NHS South West London Integrated Care Board', 'ca

In [44]:
def longitude_latitude_from_postcode(postcode):
    url = f"https://api.postcodes.io/postcodes/{postcode}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['status'] == 200:
            print(data['result'])
            return data['result']['longitude'], data['result']['latitude']
    return None, None

In [34]:
from house_functions import filter_house_df
import requests
from itertools import islice

def important_info_from_postcode(postcode):
    url = f"https://api.postcodes.io/postcodes/{postcode}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['status'] == 200:
            data = data['result']
            return data.get('longitude'), data.get('latitude'), data.get('pfa'), data.get('lsoa')
    return None, None

def get_houses_postcode(filtered_df):
    if filtered_df is not None and not filtered_df.empty:
        return set(filtered_df["Postcode"].str.upper().unique().tolist())
filtered_df, original_df = filter_house_df({"Borough": "Southwark", "Town/City": "London"})
postcode_filtered_df = filtered_df.assign(longitude=filtered_df["Postcode"].apply(important_info_from_postcode).str[0], latitude=filtered_df["Postcode"].apply(important_info_from_postcode).str[1],PFA=filtered_df["Postcode"].apply(important_info_from_postcode).str[2],LOSA=filtered_df["Postcode"].apply(important_info_from_postcode).str[3])
# postcodes = list(get_houses_postcode(filtered_df))
# n = 100
# it = iter(postcodes) 
# list_postcodes = []
# res = [list(islice(it, 100)) for _ in range((len(postcodes) + 100 - 1) // 100)]
# for postcode_batch in res:
#     payload = {"postcodes": postcode_batch}
#     res = requests.post("https://api.postcodes.io/postcodes", json=payload).json()
#     list_postcodes.extend([item['result'] for item in res.get('result', []) if item['result']])
# # # Process the results
# postcodes_info = [{'postcode': postcode_details['postcode'], 'longitude': postcode_details.get('longitude'), 'latitude': postcode_details.get('latitude'), 'lsoa': postcode_details.get('lsoa')} for postcode_details in list_postcodes]
# print(postcodes_info)
# for result in res.get('result', []):
#     if result['result']:
#         all_lsoas.append(result['result'][0]['lsoa'])

Filtering DataFrame for Borough: Southwark, Town/City: London, County: None, Postcode: None


In [35]:
postcode_filtered_df

,Unnamed: 0,Price,Date of transfer,Postcode,Property type,Door_number,first_line,Street,Town/City,Borough,County,outward_code,sector_code,longitude,latitude,PFA,LOSA
590,640,800000,2007-07-11 00:00,SE22 0SF,S,21,NaN,THERAPIA ROAD,LONDON,SOUTHWARK,GREATER LONDON,SE22,SE22 0,-0.057709,51.451782,Metropolitan Police,Southwark 029C
718,774,158500,2001-03-30 00:00,SE1 7QQ,F,GUTHRIE COURT,FLAT 27,MORLEY STREET,LONDON,SOUTHWARK,GREATER LONDON,SE1,SE1 7,-0.108638,51.499315,Metropolitan Police,Southwark 034E
840,904,520000,2021-02-12 00:00,SE22 0PA,F,269,FLAT A,UNDERHILL ROAD,LONDON,SOUTHWARK,GREATER LONDON,SE22,SE22 0,-0.067094,51.453116,Metropolitan Police,Southwark 032C
1205,1291,550000,2022-11-11 00:00,SE15 6AU,F,"SOUTH CITY COURT, 52",FLAT 326,PECKHAM GROVE,LONDON,SOUTHWARK,GREATER LONDON,SE15,SE15 6,-0.081560,51.478515,Metropolitan Police,Southwark 020B
1216,1302,385000,2022-07-13 00:00,SE15 3HB,F,139A,NaN,EVELINA ROAD,LONDON,SOUTHWARK,GREATER LONDON,SE15,SE15 3,-0.056772,51.466056,Metropolitan Police,Southwark 026B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83869,85465,800000,2026-07-01 00:00,SE5 7RW,T,20,NaN,ADA ROAD,LONDON,SOUTHWARK,GREATER LONDON,SE5,SE5 7,-0.084790,51.476698,Metropolitan Police,Southwark 020C
85488,87092,1125000,2026-06-19 00:00,SE22 9DG,T,45,NaN,NUTFIELD ROAD,LONDON,SOUTHWARK,GREATER LONDON,SE22,SE22 9,-0.074236,51.458530,Metropolitan Police,Southwark 030A
85891,87495,1350000,2026-03-31 00:00,SE22 9LF,S,31,NaN,RODWELL ROAD,LONDON,SOUTHWARK,GREATER LONDON,SE22,SE22 9,-0.073461,51.453679,Metropolitan Police,Southwark 030C
85892,87496,990000,2026-07-15 00:00,SE22 0NN,T,23,NaN,SCUTARI ROAD,LONDON,SOUTHWARK,GREATER LONDON,SE22,SE22 0,-0.057477,51.454206,Metropolitan Police,Southwark 029C


In [39]:
police_force_list = set(postcode_filtered_df["PFA"].dropna())

In [40]:
police_force_list

{'Metropolitan Police'}

In [41]:
import os 
import pandas as pd
dir_loc = "/Users/danishanrahulan/Downloads/police_force_2026_09"
police_files = []
li = []
directory = os.listdir(dir_loc)
searchstring = police_force_list
for police_force in searchstring:
    for fname in directory:
        file_path = dir_loc + os.sep + fname
        if os.path.isfile(file_path):
            f = open(dir_loc + os.sep + fname, 'r')
            if police_force in f.read():
                # print(f)
                # print(f.name())
                police_files.append(file_path)

for file in police_files:
    df = pd.read_csv(file, index_col=None, header=0)
    li.append(df)

police_force_pd = pd.concat(li, axis=0, ignore_index=True)

In [15]:
frame

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-09,Metropolitan Police Service,Metropolitan Police Service,0.134947,51.588063,On or near Mead Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
1,NaN,2026-09,Metropolitan Police Service,Metropolitan Police Service,0.140194,51.582356,On or near Hatch Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
2,NaN,2026-09,Metropolitan Police Service,Metropolitan Police Service,0.140576,51.583419,On or near Rams Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
3,NaN,2026-09,Metropolitan Police Service,Metropolitan Police Service,0.140576,51.583419,On or near Rams Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
4,NaN,2026-09,Metropolitan Police Service,Metropolitan Police Service,0.140576,51.583419,On or near Rams Grove,E01000027,Barking and Dagenham 001A,Anti-social behaviour,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
89321,475557017556cb73af7b1f6304da6771152c9e62be5098...,2026-09,Metropolitan Police Service,Metropolitan Police Service,-0.133538,51.487500,On or near Aylesford Street,E01035722,Westminster 024G,Vehicle crime,Investigation complete; no suspect identified,NaN
89322,dd1f08489e7fbe02277fe6765d9eb1a2917fc3f4abc29a...,2026-09,Metropolitan Police Service,Metropolitan Police Service,-0.135320,51.487600,On or near St George'S Square,E01035722,Westminster 024G,Violence and sexual offences,Under investigation,NaN
89323,380281b582100da98fb64c577d605e9e2ab00c490574e1...,2026-09,Metropolitan Police Service,Metropolitan Police Service,-0.137230,51.486309,On or near Parking Area,E01035722,Westminster 024G,Violence and sexual offences,Under investigation,NaN
89324,a96391a7e3bc853673a3f8f1c7c7b802339e62a12fe477...,2026-09,Metropolitan Police Service,Metropolitan Police Service,-0.133480,51.486807,On or near St George'S Square Mews,E01035722,Westminster 024G,Violence and sexual offences,Investigation complete; no suspect identified,NaN


In [41]:
import ast
ast.literal_eval(insights['amenities'])

TypeError: string indices must be integers, not 'str'

In [61]:
import numpy as np
import ast

def search_cells(lats, lons, cell_m=1500):
    dlat = cell_m / 111_320
    dlon = cell_m / (111_320 * np.cos(np.radians(np.mean(lats))))
    cells = {(int(la // dlat), int(lo // dlon)) for la, lo in zip(lats, lons)}
    return [((i + 0.5) * dlat, (j + 0.5) * dlon) for i, j in cells]

# radius per search: half the cell diagonal plus the user's distance
# radius_m = 0.71 * 1500 + max_distance_miles * 1609
# insights = ast.literal_eval(insights)
insights = ast.literal_eval(insights)
keys = insights['amenities']
area = insights['area']
cells = search_cells(postcode_filtered_df["latitude"].to_numpy(), postcode_filtered_df["longitude"].to_numpy(), cell_m=1500)
for (clat, clon) in cells:
    for item in build_search_items(keys, clat, clon, area):
        print(item)

[{'type': 'TEXT', 'key': 'morleys', 'payload': {'textQuery': "Morley's Chicken in Lambeth, London", 'locationBias': {'circle': {'center': {'latitude': 51.4799676607977, 'longitude': np.float64(-0.11892965847177742)}, 'radius': 3000.0}}, 'regionCode': 'GB', 'pageSize': 20}}, {'type': 'TEXT', 'key': 'pepes', 'payload': {'textQuery': "Pepe's Piri Piri in Lambeth, London", 'locationBias': {'circle': {'center': {'latitude': 51.4799676607977, 'longitude': np.float64(-0.11892965847177742)}, 'radius': 3000.0}}, 'regionCode': 'GB', 'pageSize': 20}}, {'type': 'TEXT', 'key': 'lidl', 'payload': {'textQuery': 'Lidl in Lambeth, London', 'locationBias': {'circle': {'center': {'latitude': 51.4799676607977, 'longitude': np.float64(-0.11892965847177742)}, 'radius': 3000.0}}, 'regionCode': 'GB', 'pageSize': 20}}, {'type': 'NEARBY', 'key': 'school', 'payload': {'includedTypes': ['primary_school', 'secondary_school'], 'locationRestriction': {'circle': {'center': {'latitude': 51.4799676607977, 'longitude': 

Once model got break down parse into what is needed we run below, where first we we start with next stage which returns a function which gives current stage and what the next thing it is running. 

router is the function where it first checks if we have a postcode matching, then runs for loops where we start with after prepare, it looks at free data in the stage and sees if the set(user amenties) so what the user asked and the stage keys keys are true, in the first case this is would be in free_data, if the user asks for an amentity , then if we have the data source. This makes sure we have something the user asked for in this stage. SO if ANY of the users amenties are in that stages lists 


In [38]:

from langchain_protocol import TypedDict
from langgraph.graph import START, END, StateGraph
class State(TypedDict):
    candidates: dict
    spec: str

In [ ]:
from map_text import AMENITIES, BRANDS, DATA_SOURCES
from email_langraph import util_st_log

STAGE_KEYS = {
    "free_data":       set(DATA_SOURCES),
    "specific_places": set(BRANDS) | {k for k, v in AMENITIES.items() if v["group"] == "cuisine"},
}
STAGE_KEYS["general_places"] = set(AMENITIES) - STAGE_KEYS["specific_places"]

ORDER = ["prepare", "free_data", "specific_places", "general_places", "report"]

class LocationGathererAgent:
    def location_check(self, state: State, max_retries: int = 3):
        location_results = {"ammenities": ["morleys", "pepes", "lidl", "good_school_ofsted", "safe"], "area": "Lambeth, London", "max_distance_miles": 0.5}
        print(f"Gathering location information for {location_results}...")

        return {"candidates": location_results, "spec":"",}

def next_stage(current):
    def router(state):
        print(f"Routing from stage: {current} with state: {state}")
        if not state["candidates"]:
            print("No candidates found, routing to no_matches")
            return "no_matches"
        for stage in ORDER[ORDER.index(current) + 1:]:
            if stage == "report" or set(state["spec"]["amenities"]) & STAGE_KEYS[stage]:
                return stage
    return router


location_gatherer = LocationGathererAgent()
builder = StateGraph(State)
builder.add_node("location_gatherer", location_gatherer.location_check)
builder.add_node("prepare", prepare_node)
builder.add_node("free_data", free_data)
builder.add_node("specific_places", specific_places)
builder.add_node("general_places", general_places)
builder.add_node("report", report)
builder.add_node("no_matches", no_matches)
builder.add_edge(START, "prepare")
builder.add_edge("report", END)
for stage in ORDER[:-1]:
    builder.add_conditional_edges(stage, next_stage(stage), ORDER[1:] + ["no_matches"])


graph = builder.compile()
graph.invoke({"spec": {"amenities": ["morleys", "pepes", "lidl", "good_school_ofsted", "safe"]}, "candidates":"Merton, London"})
# graph.invoke({"ammenities": ["morleys", "pepes", "lidl", "good_school_ofsted", "safe"],"spec":"", "area": "Lambeth, London", "max_distance_miles": 0.5})

Adding node: location_gatherer with function: <bound method LocationGathererAgent.location_check of <__main__.LocationGathererAgent object at 0x11de99400>>
current stage: prepare
calling router
current stage: free_data
calling router
current stage: specific_places
calling router
current stage: general_places
calling router
prepare_node: {'amenities': ['morleys', 'pepes', 'lidl', 'good_school_ofsted', 'safe']}
Routing from stage: prepare with state: {'candidates': 'Merton, London', 'spec': {'amenities': ['morleys', 'pepes', 'lidl', 'good_school_ofsted', 'safe']}}
Routing to next stage: free_data
free_data_node: {'amenities': ['morleys', 'pepes', 'lidl', 'good_school_ofsted', 'safe']}
Routing from stage: free_data with state: {'candidates': 'Merton, London', 'spec': {'amenities': ['morleys', 'pepes', 'lidl', 'good_school_ofsted', 'safe']}}
Routing to next stage: specific_places
specific_places: {'amenities': ['morleys', 'pepes', 'lidl', 'good_school_ofsted', 'safe']}
Routing from stage: 

{'candidates': 'Merton, London',
 'spec': {'amenities': ['morleys',
   'pepes',
   'lidl',
   'good_school_ofsted',
   'safe']}}

In [ ]:
location_dict = {"Borough": "Lambeth", "Town/City": "London"}

In [ ]:
def prepare_node(state):
    filtered_df, original_df = filter_house_df(location_dict)
    postcode_filtered_df = filtered_df.assign(longitude=filtered_df["Postcode"].apply(longitude_latitude_from_postcode).str[0], latitude=filtered_df["Postcode"].apply(longitude_latitude_from_postcode).str[1])

    return state
def general_places(state):
    print(f"general_places: {state['spec']}")
    return state
def specific_places(state):
    print(f"specific_places: {state['spec']}")
    return state
def report(state):
    print(f"report: {state['spec']}")
    return state
def no_matches(state):
    print(f"no_matches: {state['spec']}")
    return state

In [ ]:
def free_data(state):
    print(f"free_data_node: {state['spec']}")
    wanted = set(state["spec"]["amenities"]) & STAGE_KEYS["free_data"]   # e.g. {"safe", "good_school_ofsted"}
    candidates = state["candidates"]
    # if "safe" in wanted:
    #     candidates = filter_by_crime(candidates)
    # if "good_school_ofsted" in wanted:
    #     candidates = filter_by_ofsted(candidates, state["spec"]["max_distance_miles"])
    return {"candidates": candidates}

In [3]:
import time
import requests
def get_crimes_in_lat_long(lat, long, date="2025-01"):
    url = f"https://data.police.uk/api/crimes-street/all-crime?lat={lat}&lng={long}&date={date}"
    time.sleep(0.2)
    res = requests.get(url)
    return res.json()

In [4]:
year = 2024
all_crimes_year = [
    crime
    for month in range(1, 13)
    for crime in get_crimes_in_lat_long("51.412579","-0.180181", f"{year}-{month:02d}")
]

In [ ]:
"""ALL CRIMES IN YEAR FROM DF"""

In [57]:
import numpy as np
filtered_postcode_police_data = police_force_pd[police_force_pd["LSOA name"].isin(postcode_filtered_df["LOSA"])]
weights = {
    "Homicide": 10,
    "Sexual offences": 8,
    "Robbery": 7,
    "Violence with injury": 7,
    "Possession of weapons": 6,
    "Burglary": 5,
    "Violence without injury": 4,
    "Theft from person": 4,
    "Vehicle crime": 3,
    "Criminal damage arson": 3,
    "Public order": 2.5,
    "Shoplifting": 2,
    "Other theft": 2,
    "Bicycle theft": 1.5,
    "Drugs": 1.5,
    "Anti social behaviour": 1,
    "Other crime": 2,
}

filtered_postcode_police_data["severity_score"] = filtered_postcode_police_data["Crime type"].map(weights)

weighted_score = filtered_postcode_police_data["severity_score"].sum()


In [51]:
safety_score

np.float64(0.12585689750830228)

In [52]:
100 * np.exp(
    -898.5 * np.log(2) / 500
)

np.float64(28.77723728319407)

In [27]:
weighted_score

np.float64(898.5)

In [58]:
def crime_calcualtion_per_year(total_crimes_year, pop_df,lsoa_column, all_lsoas):
    print("inside crime calcuation")
    print(len(set(all_lsoas)))
    if isinstance(all_lsoas,list):
        unique_locations = {str(x).lower() for x in all_lsoas}
        filtered_po = pop_df.loc[pop_df[lsoa_column].str.lower().isin(unique_locations)]
    else:
        unique_locations = str(all_lsoas).lower()
        filtered_po = pop_df.loc[pop_df[lsoa_column].str.lower() == unique_locations]
    print("Requested LSOAs:", unique_locations)
    print("Matched rows:")
    print(filtered_po[[lsoa_column, "Total"]])
    total_populations_2024 = pd.to_numeric(filtered_po["Total"].str.replace(",", "", regex=False)).sum()
    print("Population:", total_populations_2024)
    print("Crimes weight:", total_crimes_year)
    return (total_crimes_year/total_populations_2024) *1000
population_df = pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2024 LSOA 2021-Table 1.csv",skiprows=3)
weighted_score_with_population = crime_calcualtion_per_year(weighted_score, population_df, 'LSOA 2021 Name', "Merton 012D")
safety_score = 100 * np.exp(
    -weighted_score * np.log(2) / 500
)

inside crime calcuation
11
Requested LSOAs: merton 012d
Matched rows:
      LSOA 2021 Name  Total
32297    Merton 012D  1,899
Population: 1899
Crimes weight: 4817.0


/var/folders/9s/5f93xlfn7w38b20npkj6mg300000gn/T/ipykernel_42975/1246364444.py:17: DtypeWarning: Columns (24: F19, 115: M19) have mixed types. Specify dtype option on import or set low_memory=False.
  population_df = pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2024 LSOA 2021-Table 1.csv",skiprows=3)


In [59]:
safety_score

np.float64(0.12585689750830228)

In [53]:
safety_score = 100 * np.exp(
    -473.1437598736177 * np.log(2) / 500
)

In [54]:
safety_score

np.float64(51.8966198373548)

In [21]:
postcode_filtered_df.loc[postcode_filtered_df["longitude"] == -0.180181]

,Unnamed: 0,Price,Date of transfer,Postcode,Property type,Door_number,first_line,Street,Town/City,Borough,County,outward_code,sector_code,longitude,latitude,LSOA
57,60,335000,2021-03-05 00:00,SW19 2QR,F,64,NaN,LIBERTY AVENUE,LONDON,MERTON,GREATER LONDON,SW19,SW19 2,-0.180181,51.412579,Merton 012D


In [34]:
import pandas as pd
def crime_calcualtion(total_crimes_year, pop_df,lsoa_column, all_lsoas):
    print("inside crime calcuation")
    print(len(set(all_lsoas)))
    if isinstance(all_lsoas,list):
        unique_locations = {str(x).lower() for x in all_lsoas}
        filtered_po = pop_df.loc[pop_df[lsoa_column].str.lower().isin(unique_locations)]
    else:
        unique_locations = str(all_lsoas).lower()
        filtered_po = pop_df.loc[pop_df[lsoa_column].str.lower() == unique_locations]
    print("Requested LSOAs:", unique_locations)
    print("Matched rows:")
    print(filtered_po[[lsoa_column, "Total"]])
    total_populations_2024 = pd.to_numeric(filtered_po["Total"].str.replace(",", "", regex=False)).sum()
    print("Population:", total_populations_2024)
    print("Crimes:", total_crimes_year)
    return (total_crimes_year/total_populations_2024) *1000

def population_data(year):
    if year == 2024:
        return pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2024 LSOA 2021-Table 1.csv",skiprows=3,)
    elif year == 2023:
        return pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2023 LSOA 2021-Table 1.csv",skiprows=3,)
    elif year == 2022:
        return pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2022 LSOA 2021-Table 1.csv",skiprows=3,)
    elif year == 2021:
        return pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2021 LSOA 2021-Table 1.csv",skiprows=3,)
    elif year == 2020:
        return pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2020 LSOA 2021-Table 1.csv",skiprows=3,)
    elif year == 2019:
        return pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2019 LSOA 2021-Table 1.csv",skiprows=3,)
    else:
        raise ValueError("Year not supported. Please choose from 2019, 2020, 2021, 2022, 2023, or 2024.")


In [35]:
# from house_functions import crime_calcualtion, population_data
crime_per_year_1000 = crime_calcualtion(len(all_crimes_year), population_data(year), 'LSOA 2021 Name', "Merton 012D")

inside crime calcuation
11
Requested LSOAs: merton 012d
Matched rows:
      LSOA 2021 Name  Total
32297    Merton 012D  1,899
Population: 1899
Crimes: 7500


/var/folders/9s/5f93xlfn7w38b20npkj6mg300000gn/T/ipykernel_31136/1421167086.py:21: DtypeWarning: Columns (24: F19, 115: M19) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv("/Users/danishanrahulan/Downloads/ons_population_19_24/Mid-2024 LSOA 2021-Table 1.csv",skiprows=3,)


In [36]:
crime_per_year_1000

np.float64(3949.447077409163)

In [42]:
postcode_filtered_df

,Unnamed: 0,Price,Date of transfer,Postcode,Property type,Door_number,first_line,Street,Town/City,Borough,County,outward_code,sector_code,longitude,latitude,LSOA,LSOA Code
57,60,335000,2021-03-05 00:00,SW19 2QR,F,64,NaN,LIBERTY AVENUE,LONDON,MERTON,GREATER LONDON,SW19,SW19 2,-0.180181,51.412579,Merton 012D,E01003408
385,415,180000,2004-06-04 00:00,SW18 4RW,T,40A,NaN,DAWLISH AVENUE,LONDON,MERTON,GREATER LONDON,SW18,SW18 4,-0.191974,51.439910,Merton 001E,E01003478
1030,1108,385000,2017-03-06 00:00,SW19 2LZ,F,1A,NaN,ABBEY ROAD,LONDON,MERTON,GREATER LONDON,SW19,SW19 2,-0.185926,51.414943,Merton 012A,E01003357
1575,1685,1000000,2022-02-11 00:00,SW20 9DZ,D,124,NaN,GRAND DRIVE,LONDON,MERTON,GREATER LONDON,SW20,SW20 9,-0.228887,51.402929,Merton 016E,E01003471
2110,2254,137500,1996-01-19 00:00,SW20 9NB,D,"THE COTTAGE, 172",NaN,GRAND DRIVE,LONDON,MERTON,GREATER LONDON,SW20,SW20 9,-0.227099,51.396383,Merton 020B,E01003467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83879,85475,399000,2026-07-01 00:00,SW19 3NW,F,235,GARDEN FLAT,KINGSTON ROAD,LONDON,MERTON,GREATER LONDON,SW19,SW19 3,-0.207350,51.412467,Merton 015E,E01003428
83902,85498,605000,2026-07-17 00:00,SW19 8DX,F,356,NaN,DURNSFORD ROAD,LONDON,MERTON,GREATER LONDON,SW19,SW19 8,-0.197530,51.438017,Merton 001E,E01003478
84098,85698,850000,2026-07-08 00:00,SW20 9JH,T,35,NaN,SOUTHWAY,LONDON,MERTON,GREATER LONDON,SW20,SW20 9,-0.226810,51.397898,Merton 020B,E01003467
84102,85702,575000,2026-06-30 00:00,SW20 0AQ,S,80,NaN,BEVERLEY WAY,LONDON,MERTON,GREATER LONDON,SW20,SW20 0,-0.245696,51.410412,Merton 010D,E01003445


In [ ]:
import requests

URL = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/ONSPD_NOV_2025_UK/FeatureServer/0/query"
PFA_URL = ("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
           "Police_Force_Areas_December_2023_EW_BSC/FeatureServer/0/query")
def pfa_for_lsoas(lsoa_codes, field="lsoa21cd"):
    results = {}
    for i in range(0, len(lsoa_codes), 100):          # batch to keep the query short
        batch = lsoa_codes[i:i+100]
        in_list = ",".join(f"'{c}'" for c in batch)
        params = {
            "where": f"{field} IN ({in_list}) AND doterm IS NULL",
            "outFields": f"{field},pfa23cd",
            "returnDistinctValues": "true",
            "returnGeometry": "false",
            "resultRecordCount": 2000,
            "f": "json",
        }
        data = requests.get(URL, params=params).json()
        for f in data.get("features", []):
            a = f["attributes"]
            results.setdefault(a[field], set()).add(a["pfa23cd"])
    return results
params = {
    "where": "1=1",
    "outFields": "PFA23CD,PFA23NM",
    "returnGeometry": "false",
    "f": "json",
}
data = requests.get(PFA_URL, params=params).json()
pfa_names = {f["attributes"]["PFA23CD"]: f["attributes"]["PFA23NM"]
             for f in data["features"]}

# Apply to your existing result
lsoa_to_pfa = pfa_for_lsoas(["E01003408", "E01003478"])
lsoa_to_force = {
    lsoa: [pfa_names.get(code, code) for code in codes]
    for lsoa, codes in lsoa_to_pfa.items()
}
# print(pfa_for_lsoas(["E01003408", "E01003478"]))

{'E01003408': {'E23000001'}, 'E01003478': {'E23000001'}}
